In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os

# Add parent directory to Python path
sys.path.append(os.path.abspath(".."))  

In [ ]:
# from scrna_pipeline.utils import per_sample_export

# report = per_sample_export.export_per_sample_h5ads(
#     "/Users/alahi.irfan/Desktop/Local_workspace/Data/Zheng_CancerCell_2024/merged_BCa_NAT.h5ad",
#     "/Users/alahi.irfan/Desktop/Local_workspace/Data/Zheng_CancerCell_2024//per_sample_raw",
#     sample_key="sample",      # <-- your obs column

#     filename_suffix="raw",
# )
# print(report.n_cells_per_sample)

In [ ]:
marker_dict = {
    # -----------------------
    # Epithelial 
    # -----------------------
    "Epithelial": [
        "EPCAM",   # classic epithelial marker
        "KRT8",
        "KRT18",
        "KRT19",
        "KRT17",
        "KRT5",
        "KRT14",
        "MUC1",    # luminal / glandular epithelial
    ],

    # -----------------------
    # Immune lineages
    # -----------------------
    "T_cell": [
        "CD3D", "CD3E", "CD2", "CD8A", "CD4",
    ],
    "B_cell": [
        "MS4A1", "CD79A", "CD79B",
    ],
    "Myeloid": [
        "LYZ",
        "CD68", "CD14",
        "TYROBP", "LST1",
        "CSF1R", "FCGR3A",
    ],

    # -----------------------
    # Stromal + mast
    # -----------------------
    "Endothelial": [
        "PECAM1", "VWF", "KDR",
    ],
    "Fibroblast": [
        "COL1A1", "COL1A2", "DCN", "LUM",
    ],
    "Mast": [
        "TPSAB1", "TPSB2", "CPA3", "KIT", "HDC", "MS4A2",
    ],
}

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.mixture import GaussianMixture
from anndata import AnnData


def call_tumor_like_epithelial_from_infercnv(
    adata: AnnData,
    *,
    broad_key: str = "broad_cell_type",
    epi_label: str = "Epithelial",
    cnv_key: str = "X_cnv",
    min_epi_cells: int = 200,
    use_2d: bool = True,
    bic_margin: float = 20.0,
    min_mean_gap: float = 0.02,
    tumor_p_thresh: float = 0.90,
    random_state: int = 0,
) -> pd.DataFrame:
    """
    Infer tumor-like epithelial cells using inferCNV output.

    Uses a principled 1-vs-2 component mixture model on CNV amplitude
    (optionally + CNV smoothness) within epithelial cells only.

    Adds to adata.obs:
      - cnv_amp
      - cnv_smooth
      - p_tumor_cnv
      - tumor_like_cnv

    Returns
    -------
    report : pd.DataFrame (single row)
        Diagnostic summary for this sample.
    """

    # ----------------------------
    # Checks
    # ----------------------------
    if cnv_key not in adata.obsm:
        raise KeyError(f"adata.obsm['{cnv_key}'] not found")

    if broad_key not in adata.obs:
        raise KeyError(f"adata.obs['{broad_key}'] not found")

    epi_mask = adata.obs[broad_key].astype(str).values == epi_label
    n_epi = int(epi_mask.sum())

    # Initialize outputs (safe defaults)
    adata.obs["cnv_amp"] = np.nan
    adata.obs["cnv_smooth"] = np.nan
    adata.obs["p_tumor_cnv"] = 0.0
    adata.obs["tumor_like_cnv"] = False

    if n_epi < min_epi_cells:
        return pd.DataFrame([{
            "n_epi": n_epi,
            "model": "skipped",
            "reason": f"n_epi < {min_epi_cells}",
        }])

    # ----------------------------
    # Load CNV matrix safely
    # ----------------------------
    X = adata.obsm[cnv_key]

    if sp.issparse(X):
        X = X.toarray()          # (cells × cnv_features)
    else:
        X = np.asarray(X)

    if X.ndim != 2:
        raise ValueError(
            f"{cnv_key} must be 2D (cells × features). Got shape={X.shape}"
        )

    # ----------------------------
    # Per-cell CNV features
    # ----------------------------
    cnv_amp = np.mean(np.abs(X), axis=1)

    if use_2d and X.shape[1] > 1:
        # Smoothness = contiguous CNVs → smaller adjacent diffs
        cnv_smooth = -np.mean(np.abs(np.diff(X, axis=1)), axis=1)
    else:
        cnv_smooth = np.full(X.shape[0], np.nan)
        use_2d = False

    adata.obs["cnv_amp"] = cnv_amp
    adata.obs["cnv_smooth"] = cnv_smooth

    # ----------------------------
    # Fit mixture on epithelial only
    # ----------------------------
    amp_e = cnv_amp[epi_mask]
    smooth_e = cnv_smooth[epi_mask]

    if use_2d:
        feats = np.column_stack([amp_e, smooth_e])
    else:
        feats = amp_e.reshape(-1, 1)

    g1 = GaussianMixture(
        n_components=1,
        covariance_type="full",
        random_state=random_state,
    ).fit(feats)

    g2 = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=random_state,
    ).fit(feats)

    bic1 = g1.bic(feats)
    bic2 = g2.bic(feats)
    bic_improvement = bic1 - bic2

    report = {
        "n_epi": n_epi,
        "bic_1comp": float(bic1),
        "bic_2comp": float(bic2),
        "bic_improvement": float(bic_improvement),
        "model": "1comp",
        "mean_gap": np.nan,
        "tumor_component": np.nan,
        "n_tumor_like": 0,
        "tumor_frac_epi": 0.0,
    }

    # ----------------------------
    # Decide whether to accept 2 components
    # ----------------------------
    if bic_improvement >= bic_margin:
        if use_2d:
            means = g2.means_[:, 0]   # compare on cnv_amp dimension
        else:
            means = g2.means_.ravel()

        tumor_k = int(np.argmax(means))
        normal_k = 1 - tumor_k
        mean_gap = float(means[tumor_k] - means[normal_k])

        report["mean_gap"] = mean_gap
        report["tumor_component"] = tumor_k

        if mean_gap >= min_mean_gap:
            post = g2.predict_proba(feats)[:, tumor_k]
            adata.obs.loc[epi_mask, "p_tumor_cnv"] = post
            report["model"] = "2comp"
        else:
            report["model"] = "1comp_forced (gap too small)"

    # ----------------------------
    # Final tumor-like call
    # ----------------------------
    adata.obs["tumor_like_cnv"] = adata.obs["p_tumor_cnv"].values >= tumor_p_thresh

    n_tumor = int((adata.obs["tumor_like_cnv"].values & epi_mask).sum())
    report["n_tumor_like"] = n_tumor
    report["tumor_frac_epi"] = float(n_tumor / max(n_epi, 1))

    return pd.DataFrame([report])


In [ ]:
import numpy as np
import pandas as pd

def add_infercnv_summaries(
    adata,
    *,
    cnv_layer: str = "gene_values_cnv",          # must be (n_cells, n_genes)
    chr_col: str = "chromosome",
    start_col: str = "start",
    bin_size_bp: int = 5_000_000,
    min_genes_per_region: int = 10,
    threshold: float = 0.1,
    add_bins: bool = True,
    add_discrete: bool = True,
):
    """
    Post-inferCNV summaries (per batch).
    Requires: adata.layers[cnv_layer] with shape (adata.n_obs, adata.n_vars)
    and adata.var has chromosome/start.

    Adds:
      - obs["cnv_burden"]
      - obsm["cnv_chr_mean"]
      - obsm["cnv_bin_mean_{Mb}Mb"] (optional)
      - obsm["cnv_chr_disc"] (optional)
      - obsm["cnv_bin_disc_{Mb}Mb"] (optional)
    """
    X = adata.layers[cnv_layer]
    if getattr(X, "shape", None) != (adata.n_obs, adata.n_vars):
        raise ValueError(f"layers['{cnv_layer}'] must be (n_cells, n_genes); got {getattr(X, 'shape', None)}")

    X = X.A if hasattr(X, "A") else (X.toarray() if hasattr(X, "toarray") else X)

    chrom = (
        adata.var[chr_col].astype(str)
        .str.replace("^chr", "", regex=True)
        .replace({"M": "MT", "m": "MT"})
        .values
    )

    # 1) per-cell CNV burden
    adata.obs["cnv_burden"] = np.nanmean(np.abs(X), axis=1)
   



    # 2) chromosome means
    chr_means = {}
    for c in pd.unique(chrom):
        idx = (chrom == c)
        if idx.sum() < min_genes_per_region:
            continue
        chr_means[f"chr{c}"] = X[:, idx].mean(axis=1)
    adata.obsm["cnv_chr_mean"] = pd.DataFrame(chr_means, index=adata.obs_names)

    # 3) fixed bins (optional)
    if add_bins:
        starts = pd.to_numeric(adata.var[start_col], errors="coerce").values
        ok = ~np.isnan(starts)
        starts = starts[ok].astype(np.int64)
        chrom_ok = chrom[ok]
        X_ok = X[:, ok]
        bin_ids = (starts // bin_size_bp).astype(np.int64)

        bin_means = {}
        for c in pd.unique(chrom_ok):
            c_mask = (chrom_ok == c)
            for b in np.unique(bin_ids[c_mask]):
                idx = c_mask & (bin_ids == b)
                if idx.sum() < min_genes_per_region:
                    continue
                name = f"chr{c}:{b*bin_size_bp}-{(b+1)*bin_size_bp}"
                bin_means[name] = X_ok[:, idx].mean(axis=1)

        mb = bin_size_bp // 1_000_000
        adata.obsm[f"cnv_bin_mean_{mb}Mb"] = pd.DataFrame(bin_means, index=adata.obs_names)

    # 4) discretize (optional)
    if add_discrete:
        def _disc(df):
            return (df > threshold).astype(np.int8) - (df < -threshold).astype(np.int8)

        adata.obsm["cnv_chr_disc"] = _disc(adata.obsm["cnv_chr_mean"])
        if add_bins:
            mb = bin_size_bp // 1_000_000
            adata.obsm[f"cnv_bin_disc_{mb}Mb"] = _disc(adata.obsm[f"cnv_bin_mean_{mb}Mb"])

    return adata

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt


def plot_infercnv_summaries(
    adata,
    *,
    groupby: str = "leiden",          # cluster / patient / batch / celltype
    bin_mb: int = 5,                  # must match your bin size
    cmap_cont: str = "viridis",
    cmap_div: str = "vlag",
):
    """
    Standard inferCNV visualizations.

    Requires:
      - obs["cnv_burden"]
      - obsm["cnv_chr_mean"]
      - obsm["cnv_chr_disc"]
      - optionally obsm["cnv_bin_mean_{bin_mb}Mb"]

    Produces:
      1) CNV burden on UMAP
      2) Chromosome CNV heatmap by group
      3) Genome-wide gain/loss barplot
      4) (Optional) Bin-level CNV heatmap
    """

    # --------------------------------------------------
    # 1) CNV burden on UMAP (robust scaling)
    # --------------------------------------------------
    vmin = adata.obs["cnv_burden"].quantile(0.01)
    vmax = adata.obs["cnv_burden"].quantile(0.99)

    sc.pl.umap(
        adata,
        color="cnv_burden",
        cmap=cmap_cont,
        vmin=vmin,
        vmax=vmax,
        title="CNV burden",
        show=True,
    )

    # --------------------------------------------------
    # 2) Chromosome-level CNV heatmap by group
    # --------------------------------------------------
    df_chr = adata.obsm["cnv_chr_mean"]
    avg_chr = df_chr.groupby(adata.obs[groupby]).mean()

    plt.figure(figsize=(12, 4))
    sns.heatmap(avg_chr, center=0, cmap=cmap_div)
    plt.title(f"Mean chromosome CNV by {groupby}")
    plt.xlabel("Chromosome")
    plt.ylabel(groupby)
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------
    # 3) Global CNV landscape: fraction of gain/loss
    # --------------------------------------------------
    disc = adata.obsm["cnv_chr_disc"]

    gain_frac = (disc == 1).mean(axis=0)
    loss_frac = (disc == -1).mean(axis=0)

    summary = pd.DataFrame(
        {"gain": gain_frac, "loss": loss_frac}
    ).sort_index()

    ax = summary.plot(kind="bar", figsize=(12, 4))
    ax.set_ylabel("Fraction of cells")
    ax.set_title("Genome-wide CNV gain/loss fraction")
    plt.tight_layout()
    plt.show()

    # --------------------------------------------------
    # 4) Optional: bin-level CNV heatmap
    # --------------------------------------------------
    bin_key = f"cnv_bin_mean_{bin_mb}Mb"
    if bin_key in adata.obsm:
        df_bin = adata.obsm[bin_key]
        avg_bin = df_bin.groupby(adata.obs[groupby]).mean()

        plt.figure(figsize=(16, 5))
        sns.heatmap(avg_bin, center=0, cmap=cmap_div)
        plt.title(f"Mean {bin_mb}Mb-bin CNV by {groupby}")
        plt.xlabel("Genomic bins")
        plt.ylabel(groupby)
        plt.tight_layout()
        plt.show()



In [ ]:
from anndata import AnnData
import scanpy as sc
import infercnvpy as infercnv

def show_umap_summary(name: str, adata: AnnData) -> None:



    print(f"\n### Sample: {name}")

    # rep = call_tumor_like_epithelial_from_infercnv(
    #     adata,
    #     broad_key="broad_celltype",
    #     epi_label="Epithelial",
    #     use_2d=False,
    #     bic_margin=5.0,
    #     min_mean_gap=0.01,
    #     tumor_p_thresh=0.60,
    # )
    # print(rep)
    
    # s = adata.obs["p_tumor_cnv"]
    # print("dtype:", s.dtype)
    # print("min/max:", float(np.nanmin(s)), float(np.nanmax(s)))
    # print("n<0:", int(np.sum(s.values < 0)))
    # print("unique (first 10):", pd.unique(s)[:10])





    


    sc.pl.umap(
        adata,
        color=["broad_celltype", "leiden"],
        legend_loc="on data",
        show=True,
    )
    

    # sc.pl.umap(
    #     adata,
    #     color=["tumor_like_cnv", "p_tumor_cnv"],
       
    #     show=True,
    # )
    
    infercnv.pl.chromosome_heatmap(adata, groupby="broad_celltype", dendrogram=True)
    infercnv.pl.chromosome_heatmap(adata, groupby="leiden", dendrogram=True)

    
    add_infercnv_summaries(adata, bin_size_bp=5_000_000, threshold=0.1)
    plot_infercnv_summaries(
        adata,
        groupby="leiden",   # or "sample", "patient", "broad_celltype"
    )

In [ ]:
from pathlib import Path

from scrna_pipeline.core.batch_runner import (
    BatchRunConfig,
    run_pipeline_on_batch,
    run_pipeline_on_h5ad_folder,
    constant_kwargs_factory,
)
from scrna_pipeline.pipelines.standard_per_sample import standard_per_sample_pipeline


# 2) Same pipeline kwargs for every sample
kwargs_factory = constant_kwargs_factory({
    "batch_key": "batch",           # must exist in each adata.obs
    "marker_dict": marker_dict,   # dict[str, list[str]]
   "run_infercnv": True,
   "infercnv_reference_categories": ["Myeloid"],
    'params':{
        "use_ribo_filter": False,
        "use_mito_filter": True,
    },
   
   

})

cfg = BatchRunConfig(
    out_dir=Path("/Users/alahi.irfan/Desktop/Local_workspace/Data/Chen_NatureCellBiol_2021/GSE141445_RAW/per_sample_processed"),
    strip_graph=True,      # remove neighbors graph before saving
    keep_umap=True,        # keep X_umap
    keep_pca=True,         # keep X_pca
    compression="gzip",
    verbose=True,
    strict=True,

)

saved = run_pipeline_on_h5ad_folder(
    "/Users/alahi.irfan/Desktop/Local_workspace/Data/Chen_NatureCellBiol_2021/GSE141445_RAW/per_sample_raw",
    pipeline=standard_per_sample_pipeline,
    kwargs_factory=kwargs_factory,
    config=cfg,
    delete_inputs=False,   # deletes raw only after successful save
    on_sample_done=show_umap_summary,  # or None
)